# 📡 Notebook 01 – Data Collection from arXiv

**SpaceSLM-mini** | Dataset Pipeline Step 1

This notebook walks through how we collect space-domain research abstracts
from arXiv using their public API.

---

### 🎯 What you will do in this notebook
1. Understand what arXiv is and how its API works
2. Install and configure the `arxiv` Python client
3. Query multiple space-related arXiv categories
4. Save results to a JSONL file for the next pipeline stage

### 📚 Key concepts
- **arXiv**: Open-access repository of scientific preprints
- **JSONL**: JSON Lines format – one JSON object per line, great for streaming large datasets
- **API client**: A library that wraps HTTP calls to a web service


In [ ]:
# Install dependencies if running in Colab
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'arxiv', 'tqdm', 'pyyaml', '-q'])

In [ ]:
# Change to project root (adjust if running locally)
import os
# os.chdir('/content/SpaceSLM')  # Uncomment if running in Google Colab

In [ ]:
import arxiv
import json
from pathlib import Path
from tqdm.notebook import tqdm

print(f'arxiv version: {arxiv.__version__}')
print('✅ Imports OK')

## 🔍 Understanding arXiv Categories

arXiv organises papers into subject categories. We target space-related ones:

| Category | Description |
|----------|-------------|
| `astro-ph.IM` | Instrumentation & Methods for Astrophysics |
| `astro-ph.EP` | Earth and Planetary Astrophysics |
| `physics.space-ph` | Space Physics |
| `eess.SP` | Signal Processing (satellite comms) |


In [ ]:
# Configuration
CATEGORIES = [
    'astro-ph.IM',
    'astro-ph.EP',
    'physics.space-ph',
]
MAX_RESULTS_PER_CATEGORY = 100   # start small to test
OUTPUT_FILE = Path('data/raw/arxiv_papers.jsonl')
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

print(f'Will collect from {len(CATEGORIES)} categories')
print(f'Max papers per category: {MAX_RESULTS_PER_CATEGORY}')
print(f'Output: {OUTPUT_FILE}')

In [ ]:
def collect_category(category: str, max_results: int) -> list[dict]:
    """Collect papers from a single arXiv category."""
    client = arxiv.Client(page_size=50, delay_seconds=1.0, num_retries=3)
    search = arxiv.Search(
        query=f'cat:{category}',
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
    )
    papers = []
    for result in client.results(search):
        papers.append({
            'arxiv_id':        result.entry_id.split('/')[-1],
            'title':           result.title.strip().replace('\n', ' '),
            'abstract':        result.summary.strip().replace('\n', ' '),
            'authors':         [str(a) for a in result.authors],
            'categories':      result.categories,
            'published':       result.published.isoformat() if result.published else None,
            'url':             result.pdf_url,
            'source_category': category,
        })
    return papers

print('Function defined!')

In [ ]:
# 🚀 Run collection
all_papers = []

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()  # start fresh

for category in CATEGORIES:
    print(f'\n📡 Fetching: {category}')
    papers = collect_category(category, MAX_RESULTS_PER_CATEGORY)
    
    # Save to JSONL
    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
        for p in papers:
            f.write(json.dumps(p, ensure_ascii=False) + '\n')
    
    all_papers.extend(papers)
    print(f'  ✅ Collected {len(papers)} papers')

print(f'\n🎉 Total collected: {len(all_papers)} papers')
print(f'   Saved to: {OUTPUT_FILE}')

In [ ]:
# 🔍 Inspect a sample paper
import pprint
if all_papers:
    print('Sample paper:')
    sample = {k: v for k, v in all_papers[0].items() if k != 'authors'}
    pprint.pprint(sample)

In [ ]:
# 📊 Basic statistics
from collections import Counter
import pandas as pd

df = pd.DataFrame(all_papers)
print(df.shape)
print('\nBy source category:')
print(df['source_category'].value_counts())
print(f'\nAbstract length stats (words):')
df['abstract_words'] = df['abstract'].str.split().str.len()
print(df['abstract_words'].describe().round(1))

## ✅ Summary

You have collected raw research paper abstracts from arXiv.

**Next step →** `02_dataset_preparation.ipynb`  
Clean the text, remove duplicates, and generate instruction QA pairs.
